**Imports**

In [1]:
import pandas as pd
from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, TrainingArguments, Trainer, AutoModelForCausalLM, DataCollatorForLanguageModeling
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from google.colab import drive
import torch
from transformers import BitsAndBytesConfig


In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


**Explore Dataset**

In [4]:
ds  = load_dataset("mpingale/mental-health-chat-dataset", token="HF_TOKEN")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views', 'text'],
        num_rows: 2775
    })
})


In [5]:
df_therapy = ds["train"].to_pandas()

# Keep only what we need
df_therapy = df_therapy[["questionText", "answerText"]].dropna()

print(f"Total rows: {len(df_therapy)}")
print(f"\nSample question:\n{df_therapy['questionText'].iloc[0]}")
print(f"\nSample answer:\n{df_therapy['answerText'].iloc[0]}")

Total rows: 2612

Sample question:
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?

Sample answer:
It is very common for people to have multiple issues that they want to (and need to) address in counseling.  I have had clients ask that same question and through more exploration, there is often an underlying fear that they  "can't be helped" or that they will "be too much for their therapist." I don't know if any of this rings true for you. But, most people have more than one problem in their lives and more often than not,  people have numerous significant stressors in their lives.  Let's face it, life can be complicated! Therapists are completely ready and equippe

**Formatting for Qwen Chat Template**

In [6]:
def format_conversation(row):
    return {
        "text": f"""<|im_start|>system
You are a compassionate and empathetic mental health support assistant. Respond with care and understanding.<|im_end|>
<|im_start|>user
{row['questionText']}<|im_end|>
<|im_start|>assistant
{row['answerText']}<|im_end|>"""
    }

# Apply formatting
df_formatted = df_therapy.apply(format_conversation, axis=1, result_type="expand")

print("Sample formatted conversation:")
print(df_formatted["text"].iloc[0])

Sample formatted conversation:
<|im_start|>system
You are a compassionate and empathetic mental health support assistant. Respond with care and understanding.<|im_end|>
<|im_start|>user
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?<|im_end|>
<|im_start|>assistant
It is very common for people to have multiple issues that they want to (and need to) address in counseling.  I have had clients ask that same question and through more exploration, there is often an underlying fear that they  "can't be helped" or that they will "be too much for their therapist." I don't know if any of this rings true for you. But, most people have more than one problem in their lives an

**Tokenize Dataset**

In [7]:
# Load tokenizer and model
MODEL_PATH = "/content/drive/MyDrive/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
print(f"✅ Model loaded on: {next(model.parameters()).device}")

# Split into train and test
train_df, test_df = train_test_split(df_formatted, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"Train size: {len(train_dataset)}")
print(f"Test size:  {len(test_dataset)}")

# Tokenize
def tokenize(batch):
    texts = [str(t) if t is not None else "" for t in batch["text"]]
    tokens = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
test_tokenized  = test_dataset.map(tokenize, batched=True, remove_columns=["text"])

print("✅ Tokenization done")

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0
Train size: 2350
Test size:  262


Map:   0%|          | 0/2350 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

✅ Tokenization done


**Setting Up LoRA**

In [8]:
# Prepare model for LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# → should show ~1% of parameters are trainable

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193


**Training the Model**

In [9]:
training_args = TrainingArguments(
    output_dir=MODEL_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    load_best_model_at_end=True,
    optim="paged_adamw_8bit"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM not masked LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)

trainer.train()
print("✅ Fine-tuning complete")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,2.174003,2.158927
100,2.122087,2.102330
150,2.095655,2.088053
200,2.085772,2.080779


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

✅ Fine-tuning complete


**Saving the LoRA Adapter**

In [10]:
import os

os.makedirs("/content/drive/MyDrive/Qwen2.5-3B-therapist/final", exist_ok=True)

model.save_pretrained("/content/drive/MyDrive/Qwen2.5-3B-therapist/final")
tokenizer.save_pretrained("/content/drive/MyDrive/Qwen2.5-3B-therapist/final")

print("✅ LoRA adapter saved to Drive")

✅ LoRA adapter saved to Drive


**Testing**

In [11]:
test_questions = [
    "I feel so lonely and nobody understands me",
    "I've been feeling really anxious lately and don't know why",
    "Everything feels hopeless and I don't see the point anymore"
]

print("=== Fine-tuned Qwen Responses ===\n")
for question in test_questions:
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    print(f"Q: {question}")
    print(f"A: {response}\n")

=== Fine-tuned Qwen Responses ===

Q: I feel so lonely and nobody understands me
A: I understand how you feel. It's normal to feel lonely sometimes. But remember that you are not alone. There are many people who care about you. They want to know you and understand you. It's important to talk about your feelings with someone. That person can be a family member, friend, teacher, or counselor. They can help you feel better. If you're having trouble talking to anyone, there are also online resources that can help. For example, you could use an app like BetterHelp or Talkspace. These apps connect you with therapists who can help you work through your feelings.

Q: I've been feeling really anxious lately and don't know why
A: There is no single cause of anxiety, but there are many potential sources. Some common causes include: 

  • Stressful life events (e.g., relationship problems, financial difficulties, job loss, etc.)
  • Changes in your body (e.g., hormonal changes)
  • Trauma or other

In [12]:
from google.colab import files
import os

# Save LoRA adapter locally first
model.save_pretrained("/content/lora_adapter")
tokenizer.save_pretrained("/content/lora_adapter")

# Zip it
!zip -r /content/lora_adapter.zip /content/lora_adapter

# Download to your Mac
files.download("/content/lora_adapter.zip")

  adding: content/lora_adapter/ (stored 0%)
  adding: content/lora_adapter/README.md (deflated 65%)
  adding: content/lora_adapter/tokenizer.json (deflated 81%)
  adding: content/lora_adapter/chat_template.jinja (deflated 71%)
  adding: content/lora_adapter/tokenizer_config.json (deflated 58%)
  adding: content/lora_adapter/adapter_model.safetensors (deflated 8%)
  adding: content/lora_adapter/adapter_config.json (deflated 56%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>